# State of Data Brasil —  Camada Bronze


## 1. Configuração do ambiente e conexão com a AWS

In [22]:
import boto3

BUCKET_NAME = "data-6625-3564-2976"
REGION = "us-east-1"

s3_client = boto3.client(
    "s3",
    region_name=REGION
)

try:
    s3_client.head_bucket(Bucket=BUCKET_NAME)
    print(f" Bucket encontrado: {BUCKET_NAME}")
except Exception as e:
    print(" Erro ao acessar o bucket:")
    print(e)


 Bucket encontrado: data-6625-3564-2976


## 2. Preparação da estrutura Bronze no Amazon S3
Criação de entrada e saída no S3.


In [23]:
prefixos = [
    "data-input/bronze/state-of-data/2023/",
    "data-input/bronze/state-of-data/2024/",
    "data-input/bronze/state-of-data/2025_2026/",
    "data-output/bronze/state-of-data/2023/",
    "data-output/bronze/state-of-data/2024/",
    "data-output/bronze/state-of-data/2025_2026/",
]

for prefixo in prefixos:
    s3_client.put_object(
        Bucket=BUCKET_NAME,
        Key=prefixo
    )

    print(f" Criado: s3://{BUCKET_NAME}/{prefixo}")


 Criado: s3://data-6625-3564-2976/data-input/bronze/state-of-data/2023/
 Criado: s3://data-6625-3564-2976/data-input/bronze/state-of-data/2024/
 Criado: s3://data-6625-3564-2976/data-input/bronze/state-of-data/2025_2026/
 Criado: s3://data-6625-3564-2976/data-output/bronze/state-of-data/2023/
 Criado: s3://data-6625-3564-2976/data-output/bronze/state-of-data/2024/
 Criado: s3://data-6625-3564-2976/data-output/bronze/state-of-data/2025_2026/


Arquivos csv

In [24]:
from pathlib import Path

PASTA_LOCAL = Path.cwd().parent / "kaggle"

print("Pasta dos arquivos:")
print(PASTA_LOCAL)

Pasta dos arquivos:
c:\Users\luyza\OneDrive\Desktop\state-of-data-aws\kaggle


In [25]:
arquivos = {
    "2023": {
        "local": PASTA_LOCAL / "State_of_data_BR_2023_Kaggle - df_survey_2023.csv",
        "s3": "data-input/bronze/state-of-data/2023/State_of_data_BR_2023_Kaggle - df_survey_2023.csv"
    },

    "2024": {
        "local": PASTA_LOCAL / "Final Dataset - State of Data 2024 - Kaggle - df_survey_2024.csv",
        "s3": "data-input/bronze/state-of-data/2024/Final Dataset - State of Data 2024 - Kaggle - df_survey_2024.csv"
    },

    "2025_2026": {
        "local": PASTA_LOCAL / "Final Dataset - State of Data 2025-2026 - Kaggle.csv",
        "s3": "data-input/bronze/state-of-data/2025_2026/Final Dataset - State of Data 2025-2026 - Kaggle.csv"
    }
}


In [26]:
for ano, info in arquivos.items():

    if info["local"].exists():
        tamanho_mb = info["local"].stat().st_size / (1024 * 1024)

        print(
            f" {ano}: {info['local'].name} "
            f"({tamanho_mb:.2f} MB)"
        )

    else:
        print(
            f" {ano}: arquivo não encontrado -> "
            f"{info['local']}"
        )


 2023: State_of_data_BR_2023_Kaggle - df_survey_2023.csv (14.54 MB)
 2024: Final Dataset - State of Data 2024 - Kaggle - df_survey_2024.csv (15.51 MB)
 2025_2026: Final Dataset - State of Data 2025-2026 - Kaggle.csv (9.88 MB)


### Upload dos CSVs originais

Upload dos datasets para `data-input/bronze/state-of-data/`.


In [27]:
for ano, info in arquivos.items():
    try:
        s3_client.upload_file(
            Filename=str(info["local"]),
            Bucket=BUCKET_NAME,
            Key=info["s3"]
        )

        print(f" {ano} enviado com sucesso!")
        print(f"   s3://{BUCKET_NAME}/{info['s3']}")

    except Exception as e:
        print(f" Erro ao enviar {ano}:")
        print(e)

print("Uplpad finalizado")


 2023 enviado com sucesso!
   s3://data-6625-3564-2976/data-input/bronze/state-of-data/2023/State_of_data_BR_2023_Kaggle - df_survey_2023.csv
 2024 enviado com sucesso!
   s3://data-6625-3564-2976/data-input/bronze/state-of-data/2024/Final Dataset - State of Data 2024 - Kaggle - df_survey_2024.csv
 2025_2026 enviado com sucesso!
   s3://data-6625-3564-2976/data-input/bronze/state-of-data/2025_2026/Final Dataset - State of Data 2025-2026 - Kaggle.csv
Uplpad finalizado


In [28]:
PREFIX = "data-input/bronze/state-of-data/"

response = s3_client.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=PREFIX
)
arquivos_encontrados = 0

for obj in response.get("Contents", []):

    key = obj["Key"]

    if key.endswith("/"):
        continue

    arquivos_encontrados += 1

    tamanho_mb = obj["Size"] / (1024 * 1024)

    print(f"\n {key}")
    print(f"   Tamanho: {tamanho_mb:.2f} MB")

print(f"Total de arquivos encontrados: {arquivos_encontrados}")



 data-input/bronze/state-of-data/2023/State_of_data_BR_2023_Kaggle - df_survey_2023.csv
   Tamanho: 14.54 MB

 data-input/bronze/state-of-data/2024/Final Dataset - State of Data 2024 - Kaggle - df_survey_2024.csv
   Tamanho: 15.51 MB

 data-input/bronze/state-of-data/2025_2026/Final Dataset - State of Data 2025-2026 - Kaggle.csv
   Tamanho: 9.88 MB
Total de arquivos encontrados: 3


## 3. AWS Glue Data Catalog

Criação ou validação do database `state_of_data`.


In [29]:
import boto3

AWS_REGION = "us-east-1"
DATABASE_NAME = "state_of_data"

glue_client = boto3.client(
    "glue",
    region_name=AWS_REGION
)

try:
    glue_client.get_database(
        Name=DATABASE_NAME
    )

    print(
        f" Database já existe: "
        f"{DATABASE_NAME}"
    )

except glue_client.exceptions.EntityNotFoundException:

    glue_client.create_database(
        DatabaseInput={
            "Name": DATABASE_NAME,
            "Description": (
                "Catalogo do projeto State of Data - "
                "Tech Challenge Fase 3"
            )
        }
    )

    print(
        f" Database criado: "
        f"{DATABASE_NAME}"
    )


 Database já existe: state_of_data


In [30]:
from pathlib import Path

SCRIPT_NAME = "glue-state-of-data-bronze.py"

SCRIPT_LOCAL = Path.cwd() / SCRIPT_NAME

script_bronze = r'''
import sys

from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job


args = getResolvedOptions(
    sys.argv,
    ["JOB_NAME"]
)

sc = SparkContext.getOrCreate()

glueContext = GlueContext(sc)

spark = glueContext.spark_session

job = Job(glueContext)

job.init(
    args["JOB_NAME"],
    args
)


BUCKET_NAME = "data-6625-3564-2976"

DATABASE_NAME = "state_of_data"


datasets = {

    "2023": {
        "input": (
            f"s3://{BUCKET_NAME}/"
            "data-input/bronze/state-of-data/2023/"
            "State_of_data_BR_2023_Kaggle - df_survey_2023.csv"
        ),

        "output": (
            f"s3://{BUCKET_NAME}/"
            "data-output/bronze/state-of-data/2023/"
        ),

        "table": "tb_state_data_2023_bronze"
    },

    "2024": {
        "input": (
            f"s3://{BUCKET_NAME}/"
            "data-input/bronze/state-of-data/2024/"
            "Final Dataset - State of Data 2024 - Kaggle - "
            "df_survey_2024.csv"
        ),

        "output": (
            f"s3://{BUCKET_NAME}/"
            "data-output/bronze/state-of-data/2024/"
        ),

        "table": "tb_state_data_2024_bronze"
    },

    "2025_2026": {
        "input": (
            f"s3://{BUCKET_NAME}/"
            "data-input/bronze/state-of-data/2025_2026/"
            "Final Dataset - State of Data 2025-2026 - Kaggle.csv"
        ),

        "output": (
            f"s3://{BUCKET_NAME}/"
            "data-output/bronze/state-of-data/2025_2026/"
        ),

        "table": "tb_state_data_2025_2026_bronze"
    }
}

def processar_bronze(
    ano,
    config
):

    print(f"PROCESSANDO STATE OF DATA {ano}")

    print(f"Entrada : {config['input']}")
    print(f"Saída   : {config['output']}")
    print(f"Tabela  : {config['table']}")
    frame = glueContext.create_dynamic_frame.from_options(

        connection_type="s3",

        connection_options={
            "paths": [
                config["input"]
            ],
            "recurse": False
        },

        format="csv",

        format_options={
            "withHeader": True,
            "separator": ",",
            "quoteChar": '"',
            "optimizePerformance": False
        },

        transformation_ctx=f"read_{ano}"
    )

    print(
        f" Arquivo {ano} carregado."
    )

    print(
        "Schema original:"
    )

    frame.printSchema()
    sink = glueContext.getSink(

        connection_type="s3",

        path=config["output"],

        enableUpdateCatalog=True,

        updateBehavior="UPDATE_IN_DATABASE",

        transformation_ctx=f"write_{ano}"
    )
    sink.setCatalogInfo(
        catalogDatabase=DATABASE_NAME,
        catalogTableName=config["table"]
    )

    sink.setFormat(
        "glueparquet",
        compression="snappy"
    )

    sink.writeFrame(
        frame
    )

    print(
        f" {ano} gravado em Parquet."
    )

    print(
        f" Tabela catalogada: "
        f"{DATABASE_NAME}.{config['table']}"
    )

for ano, config in datasets.items():

    processar_bronze(
        ano,
        config
    )


job.commit()

print(" CAMADA BRONZE FINALIZADA")
'''


SCRIPT_LOCAL.write_text(
    script_bronze,
    encoding="utf-8"
)

print(
    f" arquivo Glue criado localmente:"
)

print(
    SCRIPT_LOCAL
)


 arquivo Glue criado localmente:
c:\Users\luyza\OneDrive\Desktop\state-of-data-aws\notebooks\glue-state-of-data-bronze.py


### 4.1 Upload e validação do script no S3

Upload do script do Glue Job para o S3.


In [31]:
SCRIPT_S3_KEY = (
    "scripts/"
    "glue-state-of-data-bronze.py"
)
s3_client.upload_file(
    Filename=str(SCRIPT_LOCAL),
    Bucket=BUCKET_NAME,
    Key=SCRIPT_S3_KEY
)

SCRIPT_S3_URI = (
    f"s3://{BUCKET_NAME}/"
    f"{SCRIPT_S3_KEY}"
)

print(
    f" {SCRIPT_S3_URI}"
)


 s3://data-6625-3564-2976/scripts/glue-state-of-data-bronze.py


In [32]:
try:
    response = s3_client.head_object(
        Bucket=BUCKET_NAME,
        Key=SCRIPT_S3_KEY
    )

    tamanho_kb = response["ContentLength"] / 1024

    print("VALIDAÇÃO DO SCRIPT")
    print(" Script encontrado no S3!")
    print(f" {SCRIPT_S3_URI}")
    print(f" Tamanho: {tamanho_kb:.2f} KB")

except Exception as e:
    print(" Erro ao localizar script:")
    print(e)


VALIDAÇÃO DO SCRIPT
 Script encontrado no S3!
 s3://data-6625-3564-2976/scripts/glue-state-of-data-bronze.py
 Tamanho: 3.31 KB


## 5. Configuração de acesso e Glue Job

Configuração do Glue Job `glue-state-of-data-bronze`.


In [33]:
sts_client = boto3.client(
    "sts",
    region_name=AWS_REGION
)

identity = sts_client.get_caller_identity()

ACCOUNT_ID = identity["Account"]

print("IDENTIDADE AWS")

print(f"Conta: {ACCOUNT_ID}")
print(f"ARN:   {identity['Arn']}")


IDENTIDADE AWS
Conta: 662535642976
ARN:   arn:aws:sts::662535642976:assumed-role/voclabs/user5390538=luyza519@gmail.com


In [34]:
iam_client = boto3.client("iam")

try:
    response = iam_client.list_roles()

    print("\nROLES ENCONTRADAS:")

    for role in response["Roles"]:

        nome = role["RoleName"]

        if (
            "LabRole" in nome
            or "Glue" in nome
            or "service-role" in nome.lower()
        ):
            print(f" {nome}")
            print(f"   {role['Arn']}")

except Exception as e:
    print("️ Não foi possível listar as Roles via boto3.")
    print(e)



ROLES ENCONTRADAS:
 LabRole
   arn:aws:iam::662535642976:role/LabRole
 RoleForLambdaModLabRole
   arn:aws:iam::662535642976:role/RoleForLambdaModLabRole


In [35]:
JOB_NAME = "glue-state-of-data-bronze"

ROLE_ARN = (
    "arn:aws:iam::662535642976:role/LabRole"
)

SCRIPT_LOCATION = (
    "s3://data-6625-3564-2976/"
    "scripts/glue-state-of-data-bronze.py"
)

job_config = {
    "Name": JOB_NAME,

    "Role": ROLE_ARN,

    "ExecutionProperty": {
        "MaxConcurrentRuns": 1
    },

    "Command": {
        "Name": "glueetl",
        "ScriptLocation": SCRIPT_LOCATION,
        "PythonVersion": "3"
    },

    "DefaultArguments": {
        "--job-language": "python",
        "--enable-glue-datacatalog": "true",
        "--enable-continuous-cloudwatch-log": "true"
    },

    "MaxRetries": 0,
    "Timeout": 10,
    "GlueVersion": "5.0",
    "WorkerType": "G.1X",
    "NumberOfWorkers": 2
}


try:

    glue_client.get_job(
        JobName=JOB_NAME
    )

    print(
        f"️ Job {JOB_NAME} já existe."
    )

    print(
        "Atualizando configuração..."
    )

    job_update = {
        key: value
        for key, value in job_config.items()
        if key != "Name"
    }

    glue_client.update_job(
        JobName=JOB_NAME,
        JobUpdate=job_update
    )

    print(
        f" Job atualizado: {JOB_NAME}"
    )

except glue_client.exceptions.EntityNotFoundException:

    glue_client.create_job(
        **job_config
    )

    print(
        f" Job criado: {JOB_NAME}"
    )


️ Job glue-state-of-data-bronze já existe.
Atualizando configuração...
 Job atualizado: glue-state-of-data-bronze


## 6. Validações para data catalog

In [37]:
prefixos_bronze_output = [
    "data-output/bronze/state-of-data/2023/",
    "data-output/bronze/state-of-data/2024/",
    "data-output/bronze/state-of-data/2025_2026/",
]

for prefixo in prefixos_bronze_output:
    response = s3_client.list_objects_v2(
        Bucket=BUCKET_NAME,
        Prefix=prefixo
    )

    objetos = response.get("Contents", [])

    if objetos:
        s3_client.delete_objects(
            Bucket=BUCKET_NAME,
            Delete={
                "Objects": [
                    {"Key": obj["Key"]}
                    for obj in objetos
                ]
            }
        )

print("Saída Bronze anterior removida.")

Saída Bronze anterior removida.


In [38]:

import csv
import re

padrao_invalido = re.compile(r"[ ,;{}()\n\t=]")
tem_problema = False

for ano, info in arquivos.items():

    with open(
        info["local"],
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as arquivo_csv:

        reader = csv.reader(arquivo_csv)

        header = next(reader)

    colunas_invalidas = [
        coluna
        for coluna in header
        if padrao_invalido.search(coluna)
    ]

    print(f"\n Pesquisa {ano}")
    print(f"Total de colunas: {len(header)}")
    print(
        f"Colunas potencialmente problemáticas: "
        f"{len(colunas_invalidas)}"
    )

    if colunas_invalidas:

        tem_problema = True

        print("\n️ Exemplos:")

        for coluna in colunas_invalidas[:10]:
            print(f"   - {coluna}")

    else:
        print(" Nenhuma coluna problemática encontrada.")


if tem_problema:

    print(
        "️ ATENÇÃO: encontramos nomes que podem impedir "
        "a gravação em Parquet."
    )
else:

    print(
        " Headers compatíveis. Podemos executar o Glue."
    )



 Pesquisa 2023
Total de colunas: 399
Colunas potencialmente problemáticas: 399

️ Exemplos:
   - ('P0', 'id')
   - ('P1_a ', 'Idade')
   - ('P1_a_1 ', 'Faixa idade')
   - ('P1_b ', 'Genero')
   - ('P1_c ', 'Cor/raca/etnia')
   - ('P1_d ', 'PCD')
   - ('P1_e ', 'experiencia_profissional_prejudicada')
   - ('P1_e_1 ', 'Não acredito que minha experiência profissional seja afetada')
   - ('P1_e_2 ', 'Experiencia prejudicada devido a minha Cor Raça Etnia')
   - ('P1_e_3 ', 'Experiencia prejudicada devido a minha identidade de gênero')

 Pesquisa 2024
Total de colunas: 403
Colunas potencialmente problemáticas: 252

️ Exemplos:
   - 1.e.1_Não acredito que minha experiência profissional seja afetada
   - 1.e.2_Sim, devido a minha Cor/Raça/Etnia
   - 1.e.3_Sim, devido a minha identidade de gênero
   - 1.e.4_Sim, devido ao fato de ser PCD
   - 1.f.1_Quantidade de oportunidades de emprego/vagas recebidas
   - 1.f.2_Senioridade das vagas recebidas em relação à sua experiência
   - 1.f.3_Aprovação

Atualização do script Bronze no S3.

In [ ]:
from pathlib import Path

SCRIPT_NAME = "glue-state-of-data-bronze.py"
SCRIPT_LOCAL = Path.cwd() / SCRIPT_NAME

script_bronze = r'''
import sys
import re
import unicodedata

from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.dynamicframe import DynamicFrame


args = getResolvedOptions(
    sys.argv,
    ["JOB_NAME"]
)

sc = SparkContext.getOrCreate()

glueContext = GlueContext(sc)

spark = glueContext.spark_session

job = Job(glueContext)

job.init(
    args["JOB_NAME"],
    args
)


BUCKET_NAME = "data-6625-3564-2976"

DATABASE_NAME = "state_of_data"


datasets = {

    "2023": {

        "input": (
            f"s3://{BUCKET_NAME}/"
            "data-input/bronze/state-of-data/2023/"
            "State_of_data_BR_2023_Kaggle - df_survey_2023.csv"
        ),

        "output": (
            f"s3://{BUCKET_NAME}/"
            "data-output/bronze/state-of-data/2023/"
        ),

        "table": "tb_state_data_2023_bronze"
    },

    "2024": {

        "input": (
            f"s3://{BUCKET_NAME}/"
            "data-input/bronze/state-of-data/2024/"
            "Final Dataset - State of Data 2024 - Kaggle - "
            "df_survey_2024.csv"
        ),

        "output": (
            f"s3://{BUCKET_NAME}/"
            "data-output/bronze/state-of-data/2024/"
        ),

        "table": "tb_state_data_2024_bronze"
    },

    "2025_2026": {

        "input": (
            f"s3://{BUCKET_NAME}/"
            "data-input/bronze/state-of-data/2025_2026/"
            "Final Dataset - State of Data 2025-2026 - Kaggle.csv"
        ),

        "output": (
            f"s3://{BUCKET_NAME}/"
            "data-output/bronze/state-of-data/2025_2026/"
        ),

        "table": "tb_state_data_2025_2026_bronze"
    }
}

def normalizar_nome_coluna(nome):

    nome = str(nome)
    nome = unicodedata.normalize(
        "NFKD",
        nome
    )

    nome = nome.encode(
        "ascii",
        "ignore"
    ).decode(
        "ascii"
    )

    nome = nome.lower().strip()

    nome = re.sub(
        r"[^a-z0-9]+",
        "_",
        nome
    )

    nome = re.sub(
        r"_+",
        "_",
        nome
    )

    nome = nome.strip("_")

    if not nome:
        nome = "coluna"

    if nome[0].isdigit():
        nome = f"col_{nome}"

    return nome


def normalizar_colunas(df):

    nomes_originais = df.columns

    novos_nomes = []

    nomes_usados = {}

    for nome_original in nomes_originais:

        nome_base = normalizar_nome_coluna(
            nome_original
        )

        if nome_base not in nomes_usados:

            nomes_usados[nome_base] = 1
            nome_final = nome_base

        else:

            nomes_usados[nome_base] += 1

            nome_final = (
                f"{nome_base}_"
                f"{nomes_usados[nome_base]}"
            )

        novos_nomes.append(
            nome_final
        )

    return df.toDF(
        *novos_nomes
    )

def processar_bronze(
    ano,
    config
):

    print(f"PROCESSANDO {ano}")

    print(
        f"Entrada: {config['input']}"
    )

    print(
        f"Saída: {config['output']}"
    )



    dyf = glueContext.create_dynamic_frame.from_options(

        connection_type="s3",

        connection_options={
            "paths": [
                config["input"]
            ],
            "recurse": False
        },

        format="csv",

        format_options={
            "withHeader": True,
            "separator": ",",
            "quoteChar": '"'
        },

        transformation_ctx=f"read_{ano}"
    )



    df = dyf.toDF()


    print(
        f"Colunas originais: {len(df.columns)}"
    )



    df = normalizar_colunas(
        df
    )


    print(
        f"Colunas após normalização: "
        f"{len(df.columns)}"
    )


    print(
        "Exemplos de colunas:"
    )

    for coluna in df.columns[:10]:

        print(
            f" - {coluna}"
        )
    dyf_saida = DynamicFrame.fromDF(
        df,
        glueContext,
        f"dyf_saida_{ano}"
    )
    sink = glueContext.getSink(

        connection_type="s3",

        path=config["output"],

        enableUpdateCatalog=True,

        updateBehavior="UPDATE_IN_DATABASE",

        transformation_ctx=f"write_{ano}"
    )


    sink.setCatalogInfo(

        catalogDatabase=DATABASE_NAME,

        catalogTableName=config["table"]
    )


    sink.setFormat(

        "glueparquet",

        compression="snappy"
    )


    sink.writeFrame(
        dyf_saida
    )


    print(
        f" {ano} concluído."
    )

    print(
        f" Tabela: "
        f"{DATABASE_NAME}.{config['table']}"
    )


for ano, config in datasets.items():

    processar_bronze(
        ano,
        config
    )


job.commit()

print(" CAMADA BRONZE FINALIZADA")
'''


SCRIPT_LOCAL.write_text(
    script_bronze,
    encoding="utf-8"
)

print(" Novo script Bronze criado:")

print(SCRIPT_LOCAL)


 Novo script Bronze criado:
c:\Users\luyza\OneDrive\Desktop\state-of-data-aws\notebooks\glue-state-of-data-bronze.py


In [40]:
s3_client.upload_file(
    Filename=str(SCRIPT_LOCAL),
    Bucket=BUCKET_NAME,
    Key=SCRIPT_S3_KEY
)

print(" Script atualizado no S3!")

print(
    f" s3://{BUCKET_NAME}/{SCRIPT_S3_KEY}"
)


 Script atualizado no S3!
 s3://data-6625-3564-2976/scripts/glue-state-of-data-bronze.py


## 7. Execução do AWS Glue Job

In [41]:
import time
from datetime import datetime

print("EXECUÇÃO - GLUE JOB BRONZE")

print(f"Job: {JOB_NAME}")
print(f"Horário: {datetime.now()}")

try:

    response = glue_client.start_job_run(
        JobName=JOB_NAME
    )

    JOB_RUN_ID = response["JobRunId"]

    print(" Glue Job iniciado!")
    print(f"Run ID: {JOB_RUN_ID}")

except Exception as e:

    print(" Não foi possível iniciar o Glue Job.")
    print(e)

    JOB_RUN_ID = None


EXECUÇÃO - GLUE JOB BRONZE
Job: glue-state-of-data-bronze
Horário: 2026-09-14 16:52:49.085167
 Glue Job iniciado!
Run ID: jr_52b49f52407a6e9377c4e3ad7ebc4575b9dafc6514770a9704a9e4a50415e00d


In [42]:
if JOB_RUN_ID:

    estados_finais = {
        "SUCCEEDED",
        "FAILED",
        "STOPPED",
        "TIMEOUT",
        "ERROR",
        "EXPIRED"
    }

    while True:

        response = glue_client.get_job_run(
            JobName=JOB_NAME,
            RunId=JOB_RUN_ID,
            PredecessorsIncluded=False
        )

        run = response["JobRun"]

        status = run["JobRunState"]
        if status in estados_finais:
            break

        time.sleep(15)

    print("RESULTADO")

    print(f"Status: {status}")

    if status == "SUCCEEDED":

        print(" BRONZE PROCESSADA COM SUCESSO!")

    else:

        print(" Glue Job não concluiu com sucesso.")

        if run.get("ErrorMessage"):
            print("Erro:")
            print(run["ErrorMessage"])

    if run.get("ExecutionTime") is not None:
        print(
            f"Tempo de execução: "
            f"{run['ExecutionTime']} segundos"
        )


RESULTADO
Status: SUCCEEDED
 BRONZE PROCESSADA COM SUCESSO!
Tempo de execução: 141 segundos


## 8. Validação final da camada Bronze

In [43]:
PREFIX_OUTPUT = "data-output/bronze/state-of-data/"

response = s3_client.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=PREFIX_OUTPUT
)

print("VALIDAÇÃO DOS PARQUETS - BRONZE")

arquivos_por_ano = {
    "2023": [],
    "2024": [],
    "2025_2026": []
}

for obj in response.get("Contents", []):

    key = obj["Key"]

    if key.endswith("/"):
        continue

    if not key.endswith(".parquet"):
        continue

    tamanho_mb = obj["Size"] / (1024 * 1024)

    if "/2023/" in key:
        ano = "2023"

    elif "/2024/" in key:
        ano = "2024"

    elif "/2025_2026/" in key:
        ano = "2025_2026"

    else:
        continue

    arquivos_por_ano[ano].append(
        {
            "key": key,
            "size": tamanho_mb
        }
    )


for ano, arquivos_ano in arquivos_por_ano.items():

    print(f"\n {ano}")

    if not arquivos_ano:

        print(" Nenhum Parquet encontrado.")
        continue

    tamanho_total = sum(
        arquivo["size"]
        for arquivo in arquivos_ano
    )

    print(
        f" Arquivos Parquet: "
        f"{len(arquivos_ano)}"
    )

    print(
        f" Tamanho total: "
        f"{tamanho_total:.2f} MB"
    )

    for arquivo in arquivos_ano:

        nome = arquivo["key"].split("/")[-1]

        print(
            f"   - {nome} "
            f"({arquivo['size']:.2f} MB)"
        )


VALIDAÇÃO DOS PARQUETS - BRONZE

 2023
 Arquivos Parquet: 1
 Tamanho total: 1.11 MB
   - run-1789415686479-part-block-0-r-00000-snappy.parquet (1.11 MB)

 2024
 Arquivos Parquet: 1
 Tamanho total: 1.14 MB
   - run-1789415698960-part-block-0-r-00000-snappy.parquet (1.14 MB)

 2025_2026
 Arquivos Parquet: 1
 Tamanho total: 0.78 MB
   - run-1789415704566-part-block-0-r-00000-snappy.parquet (0.78 MB)


In [44]:
tabelas_esperadas = [
    "tb_state_data_2023_bronze",
    "tb_state_data_2024_bronze",
    "tb_state_data_2025_2026_bronze"
]

print("VALIDAÇÃO - GLUE DATA CATALOG")

for tabela in tabelas_esperadas:

    try:

        response = glue_client.get_table(
            DatabaseName=DATABASE_NAME,
            Name=tabela
        )

        table = response["Table"]

        colunas = table[
            "StorageDescriptor"
        ].get(
            "Columns",
            []
        )

        location = table[
            "StorageDescriptor"
        ].get(
            "Location",
            ""
        )

        print(f"\n {tabela}")

        print(
            f"   Colunas: {len(colunas)}"
        )

        print(
            f"   S3: {location}"
        )

        print("   Exemplos:")

        for coluna in colunas[:5]:

            print(
                f"      {coluna['Name']} "
                f"({coluna['Type']})"
            )

    except glue_client.exceptions.EntityNotFoundException:

        print(
            f"\n Tabela não encontrada: "
            f"{tabela}"
        )


VALIDAÇÃO - GLUE DATA CATALOG

 tb_state_data_2023_bronze
   Colunas: 399
   S3: s3://data-6625-3564-2976/data-output/bronze/state-of-data/2023/
   Exemplos:
      p0_id (string)
      p1_a_idade (string)
      p1_a_1_faixa_idade (string)
      p1_b_genero (string)
      p1_c_cor_raca_etnia (string)

 tb_state_data_2024_bronze
   Colunas: 403
   S3: s3://data-6625-3564-2976/data-output/bronze/state-of-data/2024/
   Exemplos:
      col_0_a_token (string)
      col_0_d_data_hora_envio (string)
      col_1_a_idade (string)
      col_1_a_1_faixa_idade (string)
      col_1_b_genero (string)

 tb_state_data_2025_2026_bronze
   Colunas: 388
   S3: s3://data-6625-3564-2976/data-output/bronze/state-of-data/2025_2026/
   Exemplos:
      col_0_a_token (string)
      col_0_d_data_hora_envio (string)
      col_1_a_idade (string)
      col_1_a_1_faixa_idade (string)
      col_1_b_genero (string)
